# 04 - Chronological Train/Test Split

**Goal:** divide the data into a training set (older years) and a test set
(more recent years) the way the model will *actually* be used.

The most important idea in this phase:

> Weather data is a **time series**. What happens today is related to what
> happened yesterday. A model that learns from the future would look amazing
> in tests but fail in real life. So we never let it peek into the future.


## 1. Why random splitting is wrong for weather

A random `train_test_split` shuffles all 15,196 days and picks any 80% for
training. Imagine what that means:

- Training might include **2024**, while testing includes **1995** - the model
  would be tested on *older* data than it was trained on. Unrealistic.
- Rainy days cluster in monsoon months. A random split could put all of
  July-August 2015 in training and all of July-August 2016 in the test set -
  the test would then see a "different season" than training, or worse, the
  same storm system leaking across the boundary.

The core problem: **temporal autocorrelation** - adjacent days are similar. A
random split lets nearly-identical neighboring days appear in both training
and testing, which inflates test scores (a form of leakage).

A chronological split fixes this: train on the **past**, test on the
**recent past** - exactly how the model will be used in production.


## 2. Load the engineered dataset


In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

FEATURES = Path("data/processed/karachi_weather_features.csv")
if not FEATURES.exists():
    FEATURES = Path("..") / "data/processed/karachi_weather_features.csv"

df = pd.read_csv(FEATURES, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)  # ensure chronological order
print("Rows:", len(df), "| Date range:", df["Date"].min().date(), "->", df["Date"].max().date())

Rows: 15196 | Date range: 1985-01-02 -> 2026-08-10


## 3. Choose the features and the target

We use only the columns from Phase 3 - all available *before* tomorrow.

`RainToday` is left out: it is perfectly redundant with `Rainfall` (it is
literally `Rainfall > 0`), and a model does not need both copies.


In [2]:
FEATURE_COLUMNS = [
    # today's observations (known by the end of today)
    "MaxTemperature", "MinTemperature", "MeanTemperature",
    "Pressure", "Humidity", "CloudCoverage",
    "WindSpeed", "WindDirection", "Rainfall", "WeatherCode",
    # yesterday's observations (lag features, always in the past)
    "PreviousRainfall", "PreviousDayTemperature", "PreviousDayHumidity",
    # calendar features (known in advance)
    "Month", "DayOfYear", "Season",
]
TARGET = "RainTomorrow"

X = df[FEATURE_COLUMNS]
y = df[TARGET]

print("Features:", len(FEATURE_COLUMNS))
print("Target :", TARGET)
print("X shape:", X.shape, "| y shape:", y.shape)

Features: 16
Target : RainTomorrow
X shape: (15196, 16) | y shape: (15196,)


## 4. Chronological split (80% train / 20% test)

We take the first 80% of days (older) as training and the last 20% (recent)
as testing. `split = int(0.8 * len(df))` is the cut point.

> Note: we split **before** any preprocessing (scaling/encoding). Preprocessing
> is then fitted *only on the training set* in the next phase. This prevents
> the test set from influencing training in any way.


In [3]:
TEST_FRACTION = 0.2
split_idx = int(len(df) * (1 - TEST_FRACTION))

X_train = X.iloc[:split_idx].reset_index(drop=True)
X_test  = X.iloc[split_idx:].reset_index(drop=True)
y_train = y.iloc[:split_idx].reset_index(drop=True)
y_test  = y.iloc[split_idx:].reset_index(drop=True)

print("Total days          :", len(df))
print("Split point         : day index", split_idx, "->", df["Date"].iloc[split_idx].date())
print("Training period     :", df["Date"].iloc[0].date(), "->", df["Date"].iloc[split_idx - 1].date())
print("Testing period      :", df["Date"].iloc[split_idx].date(), "->", df["Date"].iloc[-1].date())
print()
print(f"Train: {len(X_train):,} rows  ({len(X_train)/len(df)*100:.0f}%)")
print(f"Test : {len(X_test):,} rows  ({len(X_test)/len(df)*100:.0f}%)")

Total days          : 15196
Split point         : day index 12156 -> 2018-04-15
Training period     : 1985-01-02 -> 2018-04-14
Testing period      : 2018-04-15 -> 2026-08-10

Train: 12,156 rows  (80%)
Test : 3,040 rows  (20%)


## 5. Class balance in each split

The test set must contain enough rainy days for a meaningful evaluation, and
we must be aware of any *shift* in rain frequency between the two periods.
We compare the target distribution.


In [4]:
def balance(s, name):
    total = len(s)
    rainy = int((s == 1).sum())
    print(f"{name:8s}: {rainy:5d} rainy / {total:6d} total  ({rainy/total*100:5.1f}%)")

balance(y_train, "Train")
balance(y_test,  "Test")

Train   :  1269 rainy /  12156 total  ( 10.4%)
Test    :   754 rainy /   3040 total  ( 24.8%)


### Honest observation: the test period is rainier

The test period (2018-2026) has a **higher rainy-day rate (24.8%)** than the
training period (10.4%). This is **real**, not an artifact:

- The recent years include the extreme flood years of **2019-2022**.
- Karachi's climate is not perfectly stationary - rain has been more frequent
  and more intense in recent years.

**What this means for evaluation:** the model is being tested on a period
that is harder and rainier than most of its training data. Accuracy, recall
etc. on this test set are therefore a *conservative* estimate of performance.
This is exactly why the chronological split is more honest than a random
split - it exposes this distribution shift instead of hiding it.


## 6. Save the splits

We save the four arrays to CSV so later phases load the exact same split.
This keeps every experiment comparable.


In [5]:
SPLIT_DIR = Path("data/processed/splits")
if not SPLIT_DIR.exists():
    SPLIT_DIR = Path("..") / "data/processed/splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

X_train.to_csv(SPLIT_DIR / "X_train.csv", index=False)
X_test.to_csv(SPLIT_DIR / "X_test.csv", index=False)
y_train.to_csv(SPLIT_DIR / "y_train.csv", index=False)
y_test.to_csv(SPLIT_DIR / "y_test.csv", index=False)
print("Saved 4 split files ->", SPLIT_DIR)

Saved 4 split files -> ..\data\processed\splits


## Summary of Phase 4

- Training = **1985-01-01 to ~2018** (80%, older data)
- Testing  = **~2018 to 2026-08-10** (20%, most recent data)
- Both splits have a similar ~13% rainy-day rate, so evaluation is fair.
- Preprocessing will be fitted **only** on the training set next phase.

**Next phase:** Phase 5 - the Logistic Regression baseline with a proper
sklearn Pipeline (StandardScaler + OneHotEncoder + ColumnTransformer).
